In [1]:
import pandas as pd
import numpy as np
import warnings
from collections import Counter
warnings.filterwarnings("ignore")


In [2]:
feature_df = pd.read_csv("/content/drive/MyDrive/clean_retention_features_updated_file.csv")

In [3]:
feature_df = feature_df.rename(columns={
    'Department':    'MaritalStatus',
    'MaritalStatus': 'EducationField',
    'EducationField':'Department'
})

In [4]:
print("Dataset loaded. Shape:", feature_df.shape)

Dataset loaded. Shape: (1676, 27)


In [5]:
FEATURES_FOR_STRATEGY = [
    "JobSatisfaction", "WorkLifeBalance", "MonthlyIncome", "JobInvolvement",
    "TrainingTimesLastYear", "DistanceFromHome", "EnvironmentSatisfaction",
    "RelationshipSatisfaction", "PerformanceRating", "YearsSinceLastPromotion",
    "YearsInCurrentRole", "TotalWorkingYears", "JobLevel", "OverTime",
    "Age", "NumCompaniesWorked", "Education", "YearsAtCompany",
    "MaritalStatus", "Department", "BusinessTravel", "EducationField"

]

In [6]:
CATEGORICAL_FEATURES = ["MaritalStatus", "Department", "BusinessTravel", "EducationField"]

In [7]:
NUMERIC_FEATURES = [f for f in FEATURES_FOR_STRATEGY if f not in CATEGORICAL_FEATURES]

In [8]:
def build_feature_cluster_quantile(df, feature):
    values = pd.to_numeric(df[feature], errors="coerce")
    return {"q1": values.quantile(0.33), "q2": values.quantile(0.66)}

feature_models = {}
for feature in NUMERIC_FEATURES:
    feature_models[feature] = build_feature_cluster_quantile(feature_df, feature)

In [9]:
ROLE_BASED_STRATEGY_MAP = {

    "Clinical": {
        "JobSatisfaction":          {"Low": ["Immediate clinical workload review"],   "Medium": ["Peer support discussions"],       "High": ["Clinical excellence recognition"]},
        "WorkLifeBalance":          {"Low": ["Redesign hospital shift rotations"],    "Medium": ["Rotational off-days"],            "High": ["Maintain structured duty cycles"]},
        "MonthlyIncome":            {"Low": ["Clinical pay adjustment review"],       "Medium": ["Incentive allowances"],           "High": ["Performance-based bonuses"]},
        "JobInvolvement":           {"Low": ["Assign supervised case handling"],      "Medium": ["Team patient rounds"],            "High": ["Lead clinical reviews"]},
        "TrainingTimesLastYear":    {"Low": ["Mandatory clinical training"],          "Medium": ["Medical education programs"],     "High": ["Specialization sponsorship"]},
        "DistanceFromHome":         {"Low": ["Hospital accommodation support"],       "Medium": ["Shift clustering"],               "High": ["Relocation allowance"]},
        "EnvironmentSatisfaction":  {"Low": ["Improve ward conditions"],              "Medium": ["Equipment upgrade review"],       "High": ["Maintain high-quality facilities"]},
        "RelationshipSatisfaction": {"Low": ["Conflict mediation"],                   "Medium": ["Team-building rounds"],           "High": ["Interdisciplinary collaboration"]},
        "PerformanceRating":        {"Low": ["Clinical mentoring"],                   "Medium": ["Case review discussions"],        "High": ["Clinical excellence awards"]},
        "YearsSinceLastPromotion":  {"Low": ["Promotion board review"],               "Medium": ["Structured growth pathway"],      "High": ["Fast-track consultant pathway"]},
        "YearsInCurrentRole":       {"Low": ["Department rotation"],                  "Medium": ["Skill diversification"],          "High": ["Senior role enrichment"]},
        "TotalWorkingYears":        {"Low": ["Career planning"],                      "Medium": ["Retention allowance"],            "High": ["Senior retention incentives"]},
        "JobLevel":                 {"Low": ["Role evaluation review"],               "Medium": ["Responsibility expansion"],       "High": ["Clinical leadership training"]},
        "OverTime":                 {"Low": ["Reduce night shifts"],                  "Medium": ["Balanced patient allocation"],    "High": ["Burnout monitoring"]},
        "Age":                      {"Low": ["Graduate mentorship program"],          "Medium": ["Mid-career development plan"],    "High": ["Senior staff recognition"]},
        "NumCompaniesWorked":       {"Low": ["Loyalty rewards program"],              "Medium": ["Career stability planning"],      "High": ["Job-hopper engagement plan"]},
        "Education":                {"Low": ["Sponsored further education"],          "Medium": ["Advanced certification support"], "High": ["Research opportunity access"]},
        "YearsAtCompany":           {"Low": ["Onboarding buddy system"],              "Medium": ["Tenure milestone rewards"],       "High": ["Long-service recognition"]},
    },

    "Administrative": {
        "JobSatisfaction":          {"Low": ["Task redistribution"],                  "Medium": ["Feedback sessions"],              "High": ["Recognition programs"]},
        "WorkLifeBalance":          {"Low": ["Office workload balancing"],            "Medium": ["Flexible scheduling"],            "High": ["Remote work support"]},
        "MonthlyIncome":            {"Low": ["Compensation review"],                  "Medium": ["Incentive alignment"],            "High": ["Performance bonus"]},
        "JobInvolvement":           {"Low": ["Responsibility expansion"],             "Medium": ["Cross-department collaboration"], "High": ["Leadership development"]},
        "TrainingTimesLastYear":    {"Low": ["Administrative workshops"],             "Medium": ["Management training"],            "High": ["Executive training"]},
        "DistanceFromHome":         {"Low": ["Flexible timing"],                      "Medium": ["Hybrid policy"],                  "High": ["Relocation assistance"]},
        "EnvironmentSatisfaction":  {"Low": ["Office improvement"],                   "Medium": ["Workspace redesign"],             "High": ["Maintain positive culture"]},
        "RelationshipSatisfaction": {"Low": ["HR mediation"],                         "Medium": ["Team engagement"],                "High": ["Leadership networking"]},
        "PerformanceRating":        {"Low": ["Performance coaching"],                 "Medium": ["Improvement plan"],               "High": ["High-performance incentives"]},
        "YearsSinceLastPromotion":  {"Low": ["Promotion review"],                     "Medium": ["Career planning"],                "High": ["Fast-track management"]},
        "YearsInCurrentRole":       {"Low": ["Role rotation"],                        "Medium": ["Responsibility enrichment"],      "High": ["Senior management pathway"]},
        "TotalWorkingYears":        {"Low": ["Retention planning"],                   "Medium": ["Retention incentives"],           "High": ["Executive retention"]},
        "JobLevel":                 {"Low": ["Level reassessment"],                   "Medium": ["Responsibility expansion"],       "High": ["Succession planning"]},
        "OverTime":                 {"Low": ["Reduce admin overload"],                "Medium": ["Shift redistribution"],           "High": ["Monitor workload sustainability"]},
        "Age":                      {"Low": ["Graduate admin program"],               "Medium": ["Mid-career leadership path"],     "High": ["Senior advisor recognition"]},
        "NumCompaniesWorked":       {"Low": ["Loyalty recognition"],                  "Medium": ["Stability incentives"],           "High": ["Re-engagement program"]},
        "Education":                {"Low": ["Admin skills training"],                "Medium": ["Management degree support"],      "High": ["Executive education programs"]},
        "YearsAtCompany":           {"Low": ["Induction support"],                    "Medium": ["Milestone recognition"],          "High": ["Long-service benefits"]},
    },

    "Other": {
        "JobSatisfaction":          {"Low": ["One-on-one satisfaction review"],       "Medium": ["Role alignment discussion"],      "High": ["Peer recognition program"]},
        "WorkLifeBalance":          {"Low": ["Workload audit"],                       "Medium": ["Flexible hours policy"],          "High": ["Sustain current balance"]},
        "MonthlyIncome":            {"Low": ["Pay equity review"],                    "Medium": ["Performance-linked pay"],         "High": ["Retention bonus"]},
        "JobInvolvement":           {"Low": ["Assign meaningful projects"],           "Medium": ["Cross-team involvement"],         "High": ["Innovation contributor role"]},
        "TrainingTimesLastYear":    {"Low": ["General skills training"],              "Medium": ["Role-specific development"],      "High": ["Advanced learning sponsorship"]},
        "DistanceFromHome":         {"Low": ["Commute support allowance"],            "Medium": ["Hybrid work option"],             "High": ["Relocation package"]},
        "EnvironmentSatisfaction":  {"Low": ["Workspace improvement"],                "Medium": ["Environment feedback sessions"],  "High": ["Maintain positive environment"]},
        "RelationshipSatisfaction": {"Low": ["Conflict resolution support"],          "Medium": ["Team bonding activities"],        "High": ["Peer mentoring program"]},
        "PerformanceRating":        {"Low": ["Performance support plan"],             "Medium": ["Goal-setting sessions"],          "High": ["High-achiever recognition"]},
        "YearsSinceLastPromotion":  {"Low": ["Promotion eligibility review"],         "Medium": ["Career roadmap planning"],        "High": ["Fast-track consideration"]},
        "YearsInCurrentRole":       {"Low": ["Role enrichment plan"],                 "Medium": ["Lateral move opportunities"],     "High": ["Senior contributor pathway"]},
        "TotalWorkingYears":        {"Low": ["Early career mentoring"],               "Medium": ["Mid-career retention plan"],      "High": ["Senior retention package"]},
        "JobLevel":                 {"Low": ["Level advancement review"],             "Medium": ["Expanded responsibilities"],      "High": ["Leadership readiness program"]},
        "OverTime":                 {"Low": ["Overtime reduction plan"],              "Medium": ["Workload rebalancing"],           "High": ["Burnout prevention check"]},
        "Age":                      {"Low": ["Early career support"],                 "Medium": ["Career growth planning"],         "High": ["Experience retention plan"]},
        "NumCompaniesWorked":       {"Low": ["Loyalty incentive program"],            "Medium": ["Career anchoring support"],       "High": ["Engagement and belonging plan"]},
        "Education":                {"Low": ["Upskilling opportunities"],             "Medium": ["Certification support"],          "High": ["Advanced study sponsorship"]},
        "YearsAtCompany":           {"Low": ["New hire integration plan"],            "Medium": ["Tenure-based rewards"],           "High": ["Long-service appreciation"]},
    }
}

In [10]:
LOW_RISK_STRATEGY_MAP = {
    "JobSatisfaction":          {"Low": ["Conduct stay interviews to identify early dissatisfiers"],         "Medium": ["Quarterly engagement check-ins"],                         "High": ["Peer recognition and appreciation program"]},
    "WorkLifeBalance":          {"Low": ["Flexible work schedule review and adjustment"],                    "Medium": ["Optional wellness and mindfulness programs"],             "High": ["Sustain and reinforce current balance practices"]},
    "MonthlyIncome":            {"Low": ["Proactive compensation benchmarking review"],                      "Medium": ["Performance-linked pay discussion"],                      "High": ["Retention bonus planning for continued loyalty"]},
    "JobInvolvement":           {"Low": ["Assign stretch projects to increase engagement"],                  "Medium": ["Cross-team collaboration opportunities"],                 "High": ["Innovation contributor or internal champion role"]},
    "TrainingTimesLastYear":    {"Low": ["Enroll in foundational skill development programs"],               "Medium": ["Role-specific certification support"],                    "High": ["Advanced learning and conference sponsorship"]},
    "DistanceFromHome":         {"Low": ["Commute subsidy or hybrid working option"],                        "Medium": ["Flexible start and end time policy"],                     "High": ["Maintain and reinforce current arrangement"]},
    "EnvironmentSatisfaction":  {"Low": ["Workspace feedback survey and follow-up"],                         "Medium": ["Environment improvement initiatives"],                    "High": ["Maintain positive workplace environment"]},
    "RelationshipSatisfaction": {"Low": ["Team bonding and social activities"],                              "Medium": ["Peer mentoring program"],                                 "High": ["Leadership networking and visibility events"]},
    "PerformanceRating":        {"Low": ["Proactive goal-setting and coaching session"],                     "Medium": ["Bi-annual performance review and feedback"],              "High": ["High-achiever spotlight and recognition"]},
    "YearsSinceLastPromotion":  {"Low": ["Promotion eligibility and timeline discussion"],                   "Medium": ["Career roadmap and growth planning"],                     "High": ["Fast-track consideration for leadership roles"]},
    "YearsInCurrentRole":       {"Low": ["Role enrichment and challenge discussion"],                        "Medium": ["Lateral move or cross-functional opportunities"],         "High": ["Senior contributor or subject-matter expert pathway"]},
    "TotalWorkingYears":        {"Low": ["Early career mentoring and guidance program"],                     "Medium": ["Mid-career engagement and development plan"],             "High": ["Senior retention and knowledge-transfer package"]},
    "JobLevel":                 {"Low": ["Level advancement roadmap and timeline"],                          "Medium": ["Expanded responsibilities and visibility"],               "High": ["Leadership readiness and succession planning"]},
    "OverTime":                 {"Low": ["Proactive workload monitoring to prevent overload"],               "Medium": ["Workload rebalancing check-in"],                          "High": ["Burnout prevention and early warning check"]},
    "Age":                      {"Low": ["Early career development and graduate program"],                   "Medium": ["Career growth and mid-career planning"],                  "High": ["Experience retention and knowledge-sharing plan"]},
    "NumCompaniesWorked":       {"Low": ["Loyalty incentive and recognition program"],                       "Medium": ["Career anchoring and stability support"],                 "High": ["Belonging, engagement, and long-term plan"]},
    "Education":                {"Low": ["Upskilling and foundational learning opportunities"],              "Medium": ["Certification and professional development support"],      "High": ["Advanced study and postgraduate sponsorship"]},
    "YearsAtCompany":           {"Low": ["New hire integration, buddy system, and onboarding support"],      "Medium": ["Tenure milestone celebration and rewards"],               "High": ["Long-service appreciation and recognition award"]},
}

In [11]:
CATEGORICAL_STRATEGY_MAP = {

    "MaritalStatus": {
        "Single":   [
            "Offer social integration and team bonding programs",
            "Career mobility and relocation flexibility support",
            "Mentorship pairing with senior colleagues"
        ],
        "Married":  [
            "Family-friendly benefits review",
            "Flexible working hours for family commitments",
            "Review parental leave and childcare support policies"
        ],
        "Divorced": [
            "Employee assistance program (EAP) referral",
            "Financial wellness and counselling support",
            "Workload review to reduce additional stress"
        ]
    },

    "Department": {
        "Cardiology": [
            "High-stress unit psychological support program",
            "Critical care burnout prevention monitoring",
            "Cardiology-specific specialist retention package"
        ],
        "Maternity": [
            "Shift pattern review for maternity unit staff",
            "Emotional resilience and support resources",
            "Team coverage planning to reduce workload peaks"
        ],
        "Neurology": [
            "Neurology specialist retention incentive",
            "Research and publication opportunity access",
            "Advanced neurology training sponsorship"
        ]
    },

    "BusinessTravel": {
        "Travel_Frequently": [
            "Review travel frequency and workload impact",
            "Travel compensation and allowance review",
            "Offer remote working days after heavy travel periods"
        ],
        "Travel_Rarely": [
            "Maintain current low-travel balance",
            "Offer optional project travel for career exposure"
        ],
        "Non-Travel": [
            "Offer optional travel opportunities for career growth",
            "Virtual collaboration and networking programs"
        ]
    },

    "EducationField": {
        "Life Sciences": [
            "Research grant and publication support",
            "Life sciences CPD (continuing professional development) funding"
        ],
        "Medical": [
            "Medical specialisation sponsorship",
            "Clinical research participation opportunities"
        ],
        "Marketing": [
            "Healthcare marketing career development pathway",
            "Cross-functional project exposure"
        ],
        "Technical Degree": [
            "Technical skills upgrade sponsorship",
            "Innovation lab and project access"
        ],
        "Human Resources": [
            "HR leadership development program",
            "People analytics and strategy training"
        ],
        "Other": [
            "Tailored career development discussion",
            "Cross-disciplinary training opportunities"
        ]
    }
}

In [12]:
employee_role_map = dict(zip(feature_df['EmployeeID'], feature_df['JobRoleGroup']))
print("Role distribution:", Counter(employee_role_map.values()))

Role distribution: Counter({'Clinical': 1011, 'Other': 534, 'Administrative': 131})


In [13]:
def get_feature_level(feature, value):
    model = feature_models[feature]
    q1, q2 = model["q1"], model["q2"]
    value = float(value)
    if value <= q1:
        return "Low"
    elif value <= q2:
        return "Medium"
    else:
        return "High"

In [14]:
def recommend_strategies_for_employee(employee_id, job_role, shap_df, feature_df, top_n=3):
    """
    Recommends role-based, corrective strategies for HIGH-RISK employees (>= threshold).
    Uses ROLE_BASED_STRATEGY_MAP stratified by Clinical / Administrative / Other.
    """
    emp_shap = shap_df[shap_df["EmployeeID"] == employee_id].copy()
    if emp_shap.empty:
        return None

    emp_features = feature_df[feature_df["EmployeeID"] == employee_id]
    if emp_features.empty:
        return None
    emp_features = emp_features.iloc[0]

    emp_shap["abs_shap"] = emp_shap["shap_value"].abs()
    top_features = emp_shap.sort_values("abs_shap", ascending=False).head(top_n)

    results = []

    for _, row in top_features.iterrows():
        feature    = row["feature_name"]
        shap_value = row["shap_value"]

        if feature not in FEATURES_FOR_STRATEGY:
            continue

        actual_value = emp_features[feature]

        # Numeric feature
        if feature in feature_models:
            level      = get_feature_level(feature, actual_value)
            role_map   = ROLE_BASED_STRATEGY_MAP.get(job_role, {})
            strategies = role_map.get(feature, {}).get(level, [])
            stype      = "HighRisk-Numeric"

        # Categorical feature
        else:
            strategies = CATEGORICAL_STRATEGY_MAP.get(feature, {}).get(actual_value, [])
            level      = "N/A"
            stype      = "HighRisk-Categorical"

        for strategy in strategies:
            results.append({
                "EmployeeID":          employee_id,
                "RiskCategory":        "High Risk",
                "Role":                job_role,
                "StrategyType":        stype,
                "Feature":             feature,
                "FeatureValue":        actual_value,
                "ClusterLevel":        level,
                "SHAP_Impact":         shap_value,
                "RecommendedStrategy": strategy
            })

    return results if results else None

In [15]:
def recommend_strategies_low_risk(employee_id, shap_df, feature_df, top_n=3):
    """
    Recommends preventive, engagement-focused strategies for LOW-RISK employees (< threshold).
    Uses the shared LOW_RISK_STRATEGY_MAP (not role-stratified).
    """
    emp_shap = shap_df[shap_df["EmployeeID"] == employee_id].copy()
    if emp_shap.empty:
        return None

    emp_features = feature_df[feature_df["EmployeeID"] == employee_id]
    if emp_features.empty:
        return None
    emp_features = emp_features.iloc[0]

    emp_shap["abs_shap"] = emp_shap["shap_value"].abs()
    top_features = emp_shap.sort_values("abs_shap", ascending=False).head(top_n)

    results = []

    for _, row in top_features.iterrows():
        feature    = row["feature_name"]
        shap_value = row["shap_value"]

        if feature not in FEATURES_FOR_STRATEGY:
            continue

        actual_value = emp_features[feature]

        # Numeric feature
        if feature in feature_models:
            level      = get_feature_level(feature, actual_value)
            strategies = LOW_RISK_STRATEGY_MAP.get(feature, {}).get(level, [])
            stype      = "LowRisk-Numeric"

        # Categorical feature — reuse shared categorical map
        else:
            strategies = CATEGORICAL_STRATEGY_MAP.get(feature, {}).get(actual_value, [])
            level      = "N/A"
            stype      = "LowRisk-Categorical"

        for strategy in strategies:
            results.append({
                "EmployeeID":          employee_id,
                "RiskCategory":        "Low Risk",
                "Role":                "N/A",
                "StrategyType":        stype,
                "Feature":             feature,
                "FeatureValue":        actual_value,
                "ClusterLevel":        level,
                "SHAP_Impact":         shap_value,
                "RecommendedStrategy": strategy
            })

    return results if results else None

In [16]:
def recommend_for_all_employees(shap_df, feature_df, employee_role_map, top_n=3):
    all_results = []
    for emp_id in shap_df["EmployeeID"].unique():
        role       = employee_role_map.get(emp_id, "Other")
        emp_result = recommend_strategies_for_employee(emp_id, role, shap_df, feature_df, top_n)
        if emp_result:
            all_results.extend(emp_result)
    return pd.DataFrame(all_results)

In [17]:
def generate_retention_plan(risk_df, shap_df, feature_df, employee_role_map,
                             probability_threshold=0.50, top_n=2):
    """
    Generates a full retention plan for ALL employees:
      - High-risk  (>= threshold): role-based corrective strategies
      - Low-risk   (<  threshold): preventive engagement strategies
    Returns a single DataFrame with a 'RiskCategory' column to distinguish them.
    """
    high_risk_ids = risk_df[risk_df["RiskProbability"] >= probability_threshold]["EmployeeID"]
    low_risk_ids  = risk_df[risk_df["RiskProbability"] <  probability_threshold]["EmployeeID"]

    final_results = []

    # --- High-risk employees ---
    for emp_id in high_risk_ids:
        role       = employee_role_map.get(emp_id, "Other")
        strategies = recommend_strategies_for_employee(emp_id, role, shap_df, feature_df, top_n)
        if strategies:
            final_results.extend(strategies)

    # --- Low-risk employees ---
    for emp_id in low_risk_ids:
        strategies = recommend_strategies_low_risk(emp_id, shap_df, feature_df, top_n)
        if strategies:
            final_results.extend(strategies)

    return pd.DataFrame(final_results)

In [18]:
def calculate_coverage(risk_df, retention_plan, threshold=0.50):
    high_risk = risk_df[risk_df["RiskProbability"] >= threshold]["EmployeeID"].nunique()
    covered   = retention_plan[retention_plan["RiskCategory"] == "High Risk"]["EmployeeID"].nunique()
    return covered / high_risk if high_risk > 0 else 0

def calculate_low_risk_coverage(risk_df, retention_plan, threshold=0.50):
    low_risk = risk_df[risk_df["RiskProbability"] < threshold]["EmployeeID"].nunique()
    covered  = retention_plan[retention_plan["RiskCategory"] == "Low Risk"]["EmployeeID"].nunique()
    return covered / low_risk if low_risk > 0 else 0

def calculate_strategy_pool_coverage(retention_plan, role_map, cat_map, low_risk_map):
    total  = sum(len(s) for r in role_map.values() for f in r.values() for s in f.values())
    total += sum(len(s) for f in cat_map.values() for s in f.values())
    total += sum(len(s) for f in low_risk_map.values() for s in f.values())
    return retention_plan["RecommendedStrategy"].nunique() / total if total > 0 else 0

def calculate_per_employee_uniqueness(retention_plan):
    def uniqueness(group):
        return group["RecommendedStrategy"].nunique() / len(group)
    return retention_plan.groupby("EmployeeID").apply(uniqueness).mean()

def average_strategies_per_employee(retention_plan):
    n = retention_plan["EmployeeID"].nunique()
    return len(retention_plan) / n if n > 0 else 0

In [19]:
# DUMMY TEST DATA

risk_df = pd.DataFrame({
    "EmployeeID":      feature_df["EmployeeID"],
    "RiskProbability": np.random.uniform(0.3, 0.9, len(feature_df))
})

shap_rows = []
for emp_id in feature_df["EmployeeID"]:
    for feature in FEATURES_FOR_STRATEGY:
        shap_rows.append({
            "EmployeeID":   emp_id,
            "feature_name": feature,
            "shap_value":   np.random.uniform(-0.5, 0.5)
        })
shap_df = pd.DataFrame(shap_rows)
print("SHAP dataframe created:", shap_df.shape)

SHAP dataframe created: (36872, 3)


In [20]:
final_retention_plan = generate_retention_plan(
    risk_df=risk_df,
    shap_df=shap_df,
    feature_df=feature_df,
    employee_role_map=employee_role_map,
    probability_threshold=0.50,
    top_n=5
)

# Split for separate inspection
high_risk_plan = final_retention_plan[final_retention_plan["RiskCategory"] == "High Risk"]
low_risk_plan  = final_retention_plan[final_retention_plan["RiskCategory"] == "Low Risk"]


print("FINAL RETENTION PLAN SUMMARY")

print(f"Total rows            : {len(final_retention_plan)}")
print(f"High-risk rows        : {len(high_risk_plan)}")
print(f"Low-risk rows         : {len(low_risk_plan)}")
print(f"Unique employees      : {final_retention_plan['EmployeeID'].nunique()}")

print("\nStrategy type breakdown:")
print(final_retention_plan.groupby(["RiskCategory", "StrategyType"]).size().to_string())

print("RETENTION STRATEGY EVALUATION")

print(f"High-Risk Coverage            : {calculate_coverage(risk_df, final_retention_plan):.2f}")
print(f"Low-Risk Coverage             : {calculate_low_risk_coverage(risk_df, final_retention_plan):.2f}")
print(f"Strategy Pool Coverage        : {calculate_strategy_pool_coverage(final_retention_plan, ROLE_BASED_STRATEGY_MAP, CATEGORICAL_STRATEGY_MAP, LOW_RISK_STRATEGY_MAP):.2f}")
print(f"Per-Employee Uniqueness       : {calculate_per_employee_uniqueness(final_retention_plan):.2f}")
print(f"Avg Strategies per Employee   : {average_strategies_per_employee(final_retention_plan):.2f}")

print("\nSample High-Risk Plan:")
print(high_risk_plan.head(5).to_string(index=False))

print("\nSample Low-Risk Plan:")
print(low_risk_plan.head(5).to_string(index=False))

FINAL RETENTION PLAN SUMMARY
Total rows            : 10608
High-risk rows        : 7026
Low-risk rows         : 3582
Unique employees      : 1676

Strategy type breakdown:
RiskCategory  StrategyType        
High Risk     HighRisk-Categorical    2406
              HighRisk-Numeric        4620
Low Risk      LowRisk-Categorical     1268
              LowRisk-Numeric         2314
RETENTION STRATEGY EVALUATION
High-Risk Coverage            : 1.00
Low-Risk Coverage             : 1.00
Strategy Pool Coverage        : 0.89
Per-Employee Uniqueness       : 1.00
Avg Strategies per Employee   : 6.33

Sample High-Risk Plan:
 EmployeeID RiskCategory     Role         StrategyType            Feature FeatureValue ClusterLevel  SHAP_Impact                                RecommendedStrategy
    1313919    High Risk Clinical     HighRisk-Numeric YearsInCurrentRole     0.222222       Medium    -0.471677                              Skill diversification
    1313919    High Risk Clinical HighRisk-Categorical